#### Intall & Setup UV

In [ ]:
%%bash 
curl -LsSf https://astral.sh/uv/install.sh | sh

In [ ]:
#%pip install -U crewai crewai_tools 'crewai[tools, litellm]' unstructured

##### Setup and Run ollama locally

```bash

#
curl -fsSL https://ollama.com/install.sh | sh
#
sudo systemctl status ollama
sudo systemctl start ollama
sudo systemctl stop ollama

#
docker run -d -v ollama:/root/.ollama -p 11434:11434 --name ollama ollama/ollama

#
ollama list

# 4K
/set parameter num_ctx 4096
# 8K
/set parameter num_ctx 8192
# 16K
/set parameter num_ctx 16384  
# 32K
/set parameter num_ctx 32768

#
# models : sudo ls -l /usr/share/ollama/.ollama/models
#
ollama pull gemma4:latest
ollama run gemma4:latest 
ollama stop gemma4:latest

# Pull and run the latest models in one command
ollama run qwen3.5
# For the fastest local inference:
ollama run gemma4:26b
# For the latest reasoning models:
ollama run deepseek-v3.2-exp:7b
# For agentic AI with Kimi K2.6:
ollama launch kimi --model kimi-k2.6:cloud

curl -X POST http://hostmaster.sandbox.net:11434/api/generate -d '{"model": "gemma4:latest", "prompt":"Why is the sky blue?","options":{"num_ctx":4096} }'


#
# embeddings
#
ollama pull nomic-embed-text:latest
ollama run nomic-embed-text:latest
curl http://hostmaster.sandbox.net:11434/v1/embeddings -d '{"model": "nomic-embed-text:latest", "prompt": "Why is the sky blue?"}'
curl http://hostmaster.sandbox.net:11434/api/embeddings -d '{"model": "nomic-embed-text:latest", "prompt": "Why is the sky blue?"}'


# http://hostmaster.sandbox.net:11434
# http://hostmaster.sandbox.net:11434/v1/models

curl http://hostmaster.sandbox.net:11434/api/ps

curl http://hostmaster.sandbox.net:11434/api/embed -d '{
  "model": "nomic-embed-text:latest",
  "input": "Why is the sky blue?"
}'

curl http://hostmaster.sandbox.net:11434/api/show -d '{ "model": "gemma4:latest" }'


ollama rm gemma4:latest
# /usr/share/ollama/.ollama/models
curl -X DELETE http://hostmaster.sandbox.net:11434/api/delete -d '{ "model": "gemma4:latest" }'
```

```
FROM llama3

# Set parameters

PARAMETER temperature 0.8
PARAMETER stop Result

# Sets a custom system message to specify the behavior of the chat assistant

# Leaving it blank for now.

SYSTEM """"""
```
```bash
ollama create crewai-llama3 -f .\Modelfile
```

#### Seup Docker Model Runner Locally

```bash
sudo -S apt-get update
sudo -S apt-get install docker-model-plugin

docker model reinstall-runner --backend vllm --gpu cuda

docker model pull ai/gpt-oss:20B
docker model run --detach ai/gpt-oss:20B
docker model stop ai/gpt-oss:20B

docker model pull ai/smollm2
docker model run --detach ai/smollm2
docker model stop ai/smollm2

# Pull a small, efficient model (good for getting started)
docker model pull ai/llama3.2:1B-Q8_0
docker model run --detach ai/llama3.2:1B-Q8_0

# Pull a larger, more capable model
docker model pull ai/llama3.2:3B-Q4_K_M

# Pull a code-specialized model
docker model pull ai/codellama:7B-Q4_K_M

docker model pull ai/llama3.2
docker model pull ai/llama3.2:1B-Q8_0
docker model run --detach ai/llama3.2
docker model stop ai/llama3.2

# List all downloaded models with sizes
docker model list

# Show detailed information about a model
docker model inspect ai/llama3.2:1B-Q8_0

# Remove a model to free disk space
docker model rm ai/llama3.2:1B-Q8_0

# Remove all unused models
docker model prune

docker model uninstall-runner --models
docker model uninstall-runner --images

http://localhost:12434/engines/v1/models

http://vmware-ubuntu.sandbox.net:12434
# The model is now serving on a local endpoint
# Default: http://localhost:12434/v1

#
curl http://localhost:12434/engines/v1/embeddings \
  -H "Content-Type: application/json" \
  -d '{
    "model": "ai/qwen3-embedding",
    "input": "A dog is an animal"
  }'
  
curl http://localhost:12434/api/chat \
  -H "Content-Type: application/json" \
  -d '{"model": "ai/smollm2", "messages": [{"role": "user", "content": "Hello!"}]}'

# Send a chat completion request using curl
curl http://localhost:12434/engines/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{
    "model": "ai/llama3.2:1B-Q8_0",
    "messages": [
      {"role": "system", "content": "You are a helpful assistant."},
      {"role": "user", "content": "Explain Docker volumes in 3 sentences."}
    ],
    "temperature": 0.7,
    "max_tokens": 256
  }'
```

```yaml
# ./docker-compose.yaml
services:
  app:
    image: my-app
    models:
      llm:
        endpoint_var: AI_MODEL_URL
        model_var: AI_MODEL_NAME
      embedding-model:
        endpoint_var: EMBEDDING_URL
        model_var: EMBEDDING_NAME

models:
  dev_model:
    model: ai/model
    context_size: 4096
    runtime_flags:
  llm:
    model: ai/smollm2
    context_size: 4096
    runtime_flags:
      - "--temp 0.1"            # Temperature
      - "--top-p 0.9"           # Top-p sampling
      - "--verbose"             # Set verbosity level to infinity
      - "--verbose-prompt"      # Print a verbose prompt before generation
      - "--log-prefix"          # Enable prefix in log messages
      - "--log-timestamps"      # Enable timestamps in log messages
      - "--log-colors"          # Enable colored logging
  embedding-model:
    model: ai/all-minilm
    context_size: 2048
    runtime_flags:
      - "--embeddings"
```

In [10]:
#Setup config dir

%mkdir -p config

In [ ]:
# %pip install crewai crewai_tools langchain langchain_community langchain_ollama streamlit duckduckgo-search

In [ ]:
%%bash

cat <<EOF > packages/ai_module/src/notebooks/crewai/config/L001-agents.yaml
researcher:
  role: >
    {topic} Senior Data Researcher
  goal: >
    Uncover cutting-edge developments in {topic}
  backstory: >
    You're a seasoned researcher with a knack for uncovering the latest
    developments in {topic}. You find the most relevant information and
    present it clearly.
EOF


In [ ]:
%%bash

cat <<EOF > packages/ai_module/src/notebooks/crewai/config/L001-tasks.yaml
research_task:
  description: >
    Conduct thorough research about {topic}. Use web search to find current,
    credible information. The current year is 2026.
  expected_output: >
    A markdown report with clear sections: key trends, notable tools or companies,
    and implications. Aim for 800 to 1200 words. No fenced code blocks around the whole document.
  agent: researcher
  output_file: 'outputs/L001/research_task_report.md'
EOF

In [5]:
# src/latest_ai_flow/crews/content_crew/content_crew.py

from typing import List

from crewai import LLM, Agent, Crew, Process, Task
from crewai.agents.agent_builder.base_agent import BaseAgent
from crewai.project import CrewBase, agent, crew, task
from crewai_tools import SerperDevTool

@CrewBase
class ResearchCrew:
  """Single-agent research crew used inside the Flow."""

  agents: List[BaseAgent]
  tasks: List[Task]

  agents_config = "packages/ai_module/src/notebooks/crewai/config/L001-agents.yaml"
  tasks_config =  "packages/ai_module/src/notebooks/crewai/config/L001-tasks.yaml"

  @agent
  def researcher(self) -> Agent:
    return Agent(
      config=self.agents_config["researcher"],  # type: ignore[index]
      verbose=True,
      tools=[SerperDevTool()]
    )

  @task
  def research_task(self) -> Task:
    return Task(
      config=self.tasks_config["research_task"],  # type: ignore[index]
    )

  @crew
  def crew(self) -> Crew:
    return Crew(
      agents=self.agents,
      tasks=self.tasks,
      process=Process.sequential,
      verbose=True,
    )

In [6]:
# src/latest_ai_flow/main.py
import os
from pydantic import BaseModel
from crewai.flow import Flow, listen, start
#from latest_ai_flow.crews.content_crew.content_crew import ResearchCrew


class ResearchFlowState(BaseModel):
  topic: str = ""
  report: str = ""


class LatestAiFlow(Flow[ResearchFlowState]):
  @start()
  def prepare_topic(self, crewai_trigger_payload: dict | None = None):
    if crewai_trigger_payload:
      self.state.topic = crewai_trigger_payload.get("topic", "AI Agents")
    else:
      self.state.topic = "AI Agents"
    print(f"Topic: {self.state.topic}")

  @listen(prepare_topic)
  def run_research(self):
    result = ResearchCrew().crew().kickoff(inputs={"topic": self.state.topic})
    self.state.report = result.raw
    print("Research crew finished.")

  @listen(run_research)
  def summarize(self):
    print(f"Report path:{os.environ['WORK_DIR']}/outputs/L001/report.md")


def kickoff():
  LatestAiFlow().kickoff()


def plot():
  LatestAiFlow().plot()


#if __name__ == "__main__":
#  kickoff()

In [3]:
plot()

In [ ]:
kickoff()

╭─────────────────────────────────────────────── 🌊 Flow Execution ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: LatestAiFlow                                                                                             │
│  ID: 5860132a-4eed-4dc2-8d27-6dc7c55d78e9                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.3                                                                                        │
│  Latest version:  1.15.1                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🌊 Flow Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Started                                                                                                   │
│  Name: LatestAiFlow                                                                                             │
│  ID: 5860132a-4eed-4dc2-8d27-6dc7c55d78e9                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Flow started with ID: 5860132a-4eed-4dc2-8d27-6dc7c55d78e9

Topic: AI Agents


...

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: run_research                                                                                           │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.3                                                                                        │
│  Latest version:  1.15.1                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: ResearchCrew                                                                                             │
│  ID: 09f3cc18-19cb-4266-a62e-0e8d500eff8d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: research_task                                                                                            │
│  ID: ab44e4c6-e96d-42b7-9b07-21c056441f5e                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI Agents Senior Data Researcher                                                                        │
│                                                                                                                 │
│  Task: Conduct thorough research about AI Agents. Use web search to find current, credible information. The     │
│  current year is 2026.                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [ ]:
import os
from crewai import LLM, Crew, Process, Agent, Task


# Define your agents
researcher = Agent(
    role='Senior Research Analyst',
    goal='Discover new insights',
    backstory="""You're an expert at finding interesting information""",
    verbose=True,
    allow_delegation=False
)

writer = Agent(
    role='Content Writer',
    goal='Write engaging content',
    backstory="""You're a talented writer who simplifies complex information""",
    verbose=True
)

# Create tasks
research_task = Task(
    description='Find interesting facts about AI in healthcare',
    expected_output="A clear, concise summary about AI in healthcare",
    agent=researcher
)

write_task = Task(
    description='Write a short blog post about AI in healthcare',
    expected_output="Write a short blog post about AI in healthcare",
    agent=writer
)

# Form the crew
crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, write_task],
    process=Process.sequential,
    share_crew=False,
    verbose=True
)

# Execute the crew's tasks
result = crew.kickoff()

print("Here's the result:")
print(result)

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.3                                                                                        │
│  Latest version:  1.14.4                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 221c5609-1cfe-4bca-acc8-6c6b7ca3d4a9                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Find interesting facts about AI in healthcare                                                            │
│  ID: fc01df53-5deb-4eab-b1f4-465885907280                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Research Analyst                                                                                 │
│                                                                                                                 │
│  Task: Find interesting facts about AI in healthcare                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Research Analyst                                                                                 │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ## AI in Healthcare: An Analysis of Transformative Shifts Beyond Diagnosis                                     │
│                                                                                                                 │
│  The role of Artificial Intelligence in healthcare is rapidly moving past the theoretical and into the          │
│  foundational fabric of clinical practice. No longer is AI merely a diagnostic supplement; it is becoming a     │
│  critical component of operational efficiency, drug discovery, and personalized patient management. The most    │
│  significant underlying insight is that AI excels at identifying highly complex, subtle patterns within vast,   │
│  multi-modal datasets (scans, genomics, EHR notes) that exceed human pattern recognition capacity.              │
│                                                                                                                 │
│  Here is a detailed summary of the most interesting and insightful facts about AI's current and near-future     │
│  impact on medicine.                                                                                            │
│                                                                                                                 │
│  ***                                                                                                            │
│                                                                                                                 │
│  ### 🧬 1. Deep Predictive Diagnostics & Pattern Recognition                                                    │
│  AI's greatest strength is its ability to process information streams that are too large or complex for human   │
│  analysis, leading to predictive insights rather than mere diagnoses.                                           │
│                                                                                                                 │
│  *   **Ophthalmology Breakthrough:** AI models are now achieving diagnostic accuracy in diseases like           │
│  **Diabetic Retinopathy** *at or above* the level of human specialists. They can rapidly screen massive         │
│  populations via fundus images, detecting micro-changes years before they become clinically visible.            │
│  *   **Pathology Scaling:** Computational imaging allows AI tools to scan whole-slide digital pathology images  │
│  (WSI) to count specific cells or identify cancer biomarkers (like grading tumor aggressiveness) with           │
│  unprecedented speed and quantifiable consistency, reducing observer variability.                               │
│  *   **Predictive Deterioration:** In ICU settings, machine learning algorithms are used to analyze continuous  │
│  streams of vital signs (heart rate, blood pressure variability, oxygen saturation) to predict acute            │
│  events—such as septic shock or cardiac arrest—hours before traditional warning signs appear, allowing for      │
│  preemptive intervention.                                                                                       │
│                                                                                                                 │
│  ### 🧪 2. Revolutionizing Drug Discovery and Genomics                                                          │
│  AI is fundamentally rewriting the timeline and cost stru

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Writer                                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # The AI Co-Pilot: How Artificial Intelligence is Redefining the Future of Medicine                            │
│                                                                                                                 │
│  Forget the sci-fi images of robot doctors. The real revolution currently reshaping healthcare isn't            │
│  physical—it's computational. Artificial Intelligence isn't just a helpful tool; it's rapidly becoming a        │
│  co-pilot for doctors and researchers, giving human expertise superhuman power.                                 │
│                                                                                                                 │
│  For decades, medicine has been brilliant, but it has been constrained by time, data overload, and the sheer    │
│  volume of human effort required. AI promises to break those boundaries, transforming healthcare from a         │
│  reactive system (treating sickness) to a proactive, predictive, and deeply personalized science.               │
│                                                                                                                 │
│  So, how exactly is this super-smart technology changing the bedside? Here is a breakdown of the most           │
│  transformative shifts underway.                                                                                │
│                                                                                                                 │
│  ***                                                                                                            │
│                                                                                                                 │
│  ### 🔎 1. Spotting Mysteries: Diagnostics Beyond Human Sight                                                   │
│                                                                                                                 │
│  AI's greatest strength is its ability to process colossal amounts of data—genomic sequences, millions of       │
│  medical images, years of blood pressure readings—and spotting patterns that are too subtle or too numerous     │
│  for the human eye to catch.                                                                                    │
│                                                                                                                 │
│  *   **The Scanner Advantage:** In specialties like ophthalmology, AI models are already matching, or           │
│  exceeding, the accuracy of human specialists when screening for diseases like Diabetic Retinopathy. They can   │
│  flag minute warning signs years before symptoms appear.                                                        │
│  *   **Predictive Care:** Inside the Intensive Care Unit, machine learning algorithms constantly monitor vital  │
│  signs. They don't wait for a patient to crash; they predict *that a crash is coming*, alerting staff hours in  │
│  advance so interventions can happen preemptively.                                                              │
│  *   **The Deep Dive (Pathology):** AI can analyze whole-slide images of biopsies with immense speed and        │
│  consistency, highlighting biomarkers and even grading tumors with perfect, quantifiable objectivity. This      │
│  reduces human fatigue and variability, making diagnosis

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Write a short blog post about AI in healthcare                                                           │
│  Agent: Content Writer                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Here's the result:
# The AI Co-Pilot: How Artificial Intelligence is Redefining the Future of Medicine

Forget the sci-fi images of robot doctors. The real revolution currently reshaping healthcare isn't physical—it's computational. Artificial Intelligence isn't just a helpful tool; it's rapidly becoming a co-pilot for doctors and researchers, giving human expertise superhuman power.

For decades, medicine has been brilliant, but it has been constrained by time, data overload, and the sheer volume of human effort required. AI promises to break those boundaries, transforming healthcare from a reactive system (treating sickness) to a proactive, predictive, and deeply personalized science.

So, how exactly is this super-smart technology changing the bedside? Here is a breakdown of the most transformative shifts underway.

***

### 🔎 1. Spotting Mysteries: Diagnostics Beyond Human Sight

AI's greatest strength is its ability to process colossal amounts of data—genomic sequences, millions o

╭────────────────────────── Trace Batch Finalization ──────────────────────────╮
│ ✅ Trace batch finalized with session ID:                                    │
│ 48b44890-0550-4598-b761-40a891af2d2b                                         │
│                                                                              │
│ 🔗 View here:                                                                │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/48b44890-0550-459 │
│ 8-b761-40a891af2d2b?access_code=TRACE-9a3dda8d6a                             │
│ 🔑 Access Code: TRACE-9a3dda8d6a                                             │
╰──────────────────────────────────────────────────────────────────────────────╯


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 221c5609-1cfe-4bca-acc8-6c6b7ca3d4a9                                                                       │
│  Final Output: # The AI Co-Pilot: How Artificial Intelligence is Redefining the Future of Medicine              │
│                                                                                                                 │
│  Forget the sci-fi images of robot doctors. The real revolution currently reshaping healthcare isn't            │
│  physical—it's computational. Artificial Intelligence isn't just a helpful tool; it's rapidly becoming a        │
│  co-pilot for doctors and researchers, giving human expertise superhuman power.                                 │
│                                                                                                                 │
│  For decades, medicine has been brilliant, but it has been constrained by time, data overload, and the sheer    │
│  volume of human effort required. AI promises to break those boundaries, transforming healthcare from a         │
│  reactive system (treating sickness) to a proactive, predictive, and deeply personalized science.               │
│                                                                                                                 │
│  So, how exactly is this super-smart technology changing the bedside? Here is a breakdown of the most           │
│  transformative shifts underway.                                                                                │
│                                                                                                                 │
│  ***                                                                                                            │
│                                                                                                                 │
│  ### 🔎 1. Spotting Mysteries: Diagnostics Beyond Human Sight                                                   │
│                                                                                                                 │
│  AI's greatest strength is its ability to process colossal amounts of data—genomic sequences, millions of       │
│  medical images, years of blood pressure readings—and spotting patterns that are too subtle or too numerous     │
│  for the human eye to catch.                                                                                    │
│                                                                                                                 │
│  *   **The Scanner Advantage:** In specialties like ophthalmology, AI models are already matching, or           │
│  exceeding, the accuracy of human specialists when screening for diseases like Diabetic Retinopathy. They can   │
│  flag minute warning signs years before symptoms appear.                                                        │
│  *   **Predictive Care:** Inside the Intensive Care Unit, machine learning algorithms constantly monitor vital  │
│  signs. They don't wait for a patient to crash; they predict *that a crash is coming*, alerting staff hours in  │
│  advance so interventions can happen preemptively.                                                              │
│  *   **The Deep Dive (Pathology):** AI can analyze whole-slide images of biopsies with immense speed and        │
│  consistency, highlighting biomarkers and even grading tumors with perfect, quantifiable objectivity. This      │
│  reduces human fatigue and variability, making diagnosi

In [5]:
from sentence_transformers import SentenceTransformer

# 1. Load a pretrained Sentence Transformer model
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
# The sentences to encode
sentences = [
    "The weather is lovely today.",
    "It's so sunny outside!",
    "He drove to the stadium.",
]

# 2. Calculate embeddings by calling model.encode()
embeddings = model.encode(sentences)
print(embeddings.shape)
# [3, 384]

# 3. Calculate the embedding similarities
similarities = model.similarity(embeddings, embeddings)
print(similarities)
# tensor([[1.0000, 0.6660, 0.1046],
#         [0.6660, 1.0000, 0.1411],
#         [0.1046, 0.1411, 1.0000]])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

(3, 768)
tensor([[1.0000, 0.6817, 0.0492],
        [0.6817, 1.0000, 0.0421],
        [0.0492, 0.0421, 1.0000]])


In [4]:
import os
from crewai import LLM
from pydantic import BaseModel

class Dog(BaseModel):
    name: str
    age: int
    breed: str

ollama_llm = LLM(
  model=f"ollama/{os.environ["OPENAI_MODEL_NAME"]}",
  base_url=os.environ["OLLAMA_URL"],
  temperature=0.8,
  response_format=Dog,
  api_key="ollama"  # Ollama doesn't require a real API key
)

response = ollama_llm.call(
    "Analyze the following messages and return the name, age, and breed. "
    "Meet Kona! She is 3 years old and is a black german shepherd."
)
print(response)

name='Kona' age=3 breed='German Shepherd'


In [ ]:
import asyncio
from crewai import LLM

async def stream_async():
    llm = LLM(
        model=f"ollama/{os.environ["OPENAI_MODEL_NAME"]}",
        base_url=os.environ["OLLAMA_URL"],
        temperature=0.8,
        stream=True,
        api_key="ollama"  # Ollama doesn't require a real API key
    )
    response = await llm.acall("Write a short story about AI")
    print(response)

#asyncio.run(stream_async())





In [5]:
await stream_async()

Output()

RecursionError: maximum recursion depth exceeded

In [ ]:
import nest_asyncio
nest_asyncio.apply()
asyncio.run(stream_async()) # Now will work


In [ ]:
#
loop = asyncio.get_running_loop()
loop.create_task(stream_async())

In [ ]:
# To execute a new event loop without interfering with the current one, you can run your async code in a background thread using ThreadPoolExecutor.
from concurrent.futures import ThreadPoolExecutor
with ThreadPoolExecutor() as executor:
    result = executor.submit(lambda: asyncio.run(stream_async())).result()

In [3]:
#
import os
import asyncio
from crewai import LLM

async def main():
    llm = LLM(
        model=f"ollama/{os.environ["OPENAI_MODEL_NAME"]}",
        base_url=os.environ["OLLAMA_URL"],
        temperature=0.8,
        response_format=Dog,
        stream=True,
        api_key="ollama"  # Ollama doesn't require a real API key
    )

    # Single async call
    response = await llm.acall("What is the capital of France?")
    print(response)

asyncio.run(main())

RuntimeError: asyncio.run() cannot be called from a running event loop

In [ ]:
from crewai.rag.config.utils import set_rag_config, get_rag_client, clear_rag_config
from chromadb.utils.embedding_functions.ollama_embedding_function import OllamaEmbeddingFunction
from crewai.rag.chromadb.types import ChromaEmbeddingFunctionWrapper
from typing import Literal, cast

# ChromaDB (default)
from crewai.rag.chromadb.config import ChromaDBConfig



set_rag_config(ChromaDBConfig(
    embedding_function=cast(
        ChromaEmbeddingFunctionWrapper,
        OllamaEmbeddingFunction(
            model_name="nomic-embed-text:latest",
            url="http://hostmaster.sandbox.net:11434/api/embeddings"
        ),
    )
))

chromadb_client = get_rag_client()

# Qdrant
#from crewai.rag.qdrant.config import QdrantConfig
#set_rag_config(QdrantConfig())
#qdrant_client = get_rag_client()

# Example operations (same API for any provider)
client = chromadb_client  # or chromadb_client
client.create_collection(collection_name="docs")
client.add_documents(
    collection_name="docs",
    documents=[{"id": "1", "content": "CrewAI enables collaborative AI agents."}],
)
results = client.search(collection_name="docs", query="collaborative agents", limit=3)

clear_rag_config()  # optional reset

In [ ]:
from crewai_tools import PDFSearchTool

# - embedding_model (required): choose provider + provider-specific config
# - vectordb (required): choose vector DB and pass its config

tool = PDFSearchTool(
    pdf='knowledge/pdf/2026-04-01_BRIJESHD_PROFILE_MP.pdf',
    config={
        "embedding_model": {
            # Supported providers: "openai", "azure", "google-generativeai", "google-vertex",
            # "voyageai", "cohere", "huggingface", "jina", "sentence-transformer",
            # "text2vec", "ollama", "openclip", "instructor", "onnx", "roboflow", "watsonx", "custom"
            "provider": "openai",  # or: "google-generativeai", "cohere", "ollama", ...
            "config": {
                # Model identifier for the chosen provider. "model" will be auto-mapped to "model_name" internally.
                "model": "nomic-embed-text:latest",
                # Optional: API key. If omitted, the tool will use provider-specific env vars
                # (e.g., OPENAI_API_KEY or EMBEDDINGS_OPENAI_API_KEY for OpenAI).
                # "api_key": "sk-...",

                # Provider-specific examples:
                # --- Google Generative AI ---
                # (Set provider="google-generativeai" above)
                # "model_name": "gemini-embedding-001",
                # "task_type": "RETRIEVAL_DOCUMENT",
                # "title": "Embeddings",

                # --- Cohere ---
                # (Set provider="cohere" above)
                # "model": "embed-english-v3.0",

                # --- Ollama (local) ---
                # (Set provider="ollama" above)
                # "model": "nomic-embed-text",
            },
        },
        "vectordb": {
                    "provider": "chromadb",  # or "qdrant"
                    "config": {
                        # For ChromaDB: pass "settings" (chromadb.config.Settings) or rely on defaults.
                        # Example (uncomment and import):
                        # from chromadb.config import Settings
                        # "settings": Settings(
                        #     persist_directory="/content/chroma",
                        #     allow_reset=True,
                        #     is_persistent=True,
                        # ),

                        # For Qdrant: pass "vectors_config" (qdrant_client.models.VectorParams).
                        # Example (uncomment and import):
                        # from qdrant_client.models import VectorParams, Distance
                        # "vectors_config": VectorParams(size=384, distance=Distance.COSINE),

                        # Note: collection name is controlled by the tool (default: "rag_tool_collection"), not set here.
                    }
        },
    }
)


CREWAI_TOOLS_ALLOW_UNSAFE_PATHS is enabled — skipping file path validation for: /home/brijeshdhaker/IdeaProjects/bd-notebooks-module/knowledge/pdf/2026-04-01_BRIJESHD_PROFILE_MP.pdf


In [5]:
tool.run("What are the skills ?")

"Relevant Content:\n\nimproved release frequency from monthly to bi-weekly, ensuring on-time delivery against stringent regulatory \n\ndeadlines. \n\n \n\n• Led Data Engineering Chapter for IB Markets Regulatory Reporting, governing 5+ regulatory \n\nframeworks (MIFID-II, RTS23, EUBR/UKBR, SSD, SFTR) — overseeing ingestion, reconciliation, and distribution \n\nof deal-store data for ~10 million+ daily trade & transaction records with full audit trail and zero regulatory \n\nbreach incidents across a 3-year delivery window. \n\n \n\n\n\n\n\n\nPage 2:\n\n \n\n• Designed and executed migration strategy from Cloudera Data Platform to Azure Databricks, leveraging Azure \n\nData Factory, ADX, and Apache Flink — reducing data processing latency by ~50%, cutting infrastructure \n\nlicensing costs by ~35%, and enabling elastic scaling for peak regulatory reporting windows without manual \n\nintervention. \n\n \n\n• Partnered with business, operations, and risk teams to identify and remediate 3 

In [1]:
import os
from openai import OpenAI

client = OpenAI(
    base_url=os.environ["OPENAI_API_BASE"],
    api_key='ollama', # Required by SDK but ignored locally
)

In [2]:
response = client.embeddings.create(
    model="nomic-embed-text:latest",
    input="This is a test sentence to generate embeddings."
)

# Access the vector
embedding_vector = response.data[0].embedding
print(embedding_vector)

[0.00924593023955822, 0.04528261721134186, -0.16524161398410797, -0.05031146854162216, 0.0739152729511261, -0.0394742488861084, 0.04313960298895836, -0.01628810167312622, -0.018023701384663582, -0.07231346517801285, -0.018154777586460114, 0.032760512083768845, 0.06767623126506805, 0.05999815836548805, -0.035834524780511856, -0.009172101505100727, 0.033963922411203384, -0.07278915494680405, -0.052704114466905594, 0.06138024106621742, -0.027492057532072067, 0.01732649840414524, 0.0016753139207139611, -0.03225832059979439, 0.09240402281284332, 0.007276293821632862, 0.029320890083909035, 0.01257240679115057, -0.025617143139243126, 0.007536767516285181, 0.016709143295884132, -0.008847963064908981, 0.005388629622757435, 0.001366117619909346, -0.07331203669309616, -0.023065969347953796, 0.0010582971153780818, 0.04340774938464165, -0.006726095452904701, -0.034795086830854416, -0.0039733704179525375, 0.04191763699054718, -0.031100422143936157, -0.014954803511500359, 0.004182626958936453, 0.0021